In [0]:
from pyspark.sql import functions as F

MY_ID = "arzoo"
VOL = f"/Volumes/workspace/capstone_{MY_ID}/raw"
S   = f"workspace.capstone_{MY_ID}"

In [0]:
b = spark.table(f"{S}.bronze_complaints")
w = spark.table(f"{S}.bronze_wards").select("ward_id", "zone")

# Decision 1: dedupe on complaint_id
d = b.dropDuplicates(["complaint_id"])
print("after dedupe:", d.count())                       # 60,500

# Decision 1: try_cast, never cast
t = (d.withColumnRenamed("closed_ts", "closed_raw")
       .withColumn("raised_ts", F.expr("try_cast(raised_ts as timestamp)"))
       .withColumn("closed_ts", F.expr("try_cast(closed_raw as timestamp)")))
print("closed_ts lost to NULL:",
      t.filter("closed_raw = 'NA' AND closed_ts IS NULL").count())   # 1,200

# Decision 2: keep the two kinds of NULL apart
t = t.withColumn("closure_known", F.expr("status = 'CLOSED' AND closed_ts IS NOT NULL"))

# Decision 3: reject reasons (the slices are disjoint, so order doesn't change counts)
t = (t.join(w, "ward_id", "left")
      .withColumn("reject_reason", F.expr("""
        CASE WHEN zone IS NULL THEN 'unknown_ward'
             WHEN raised_ts IS NULL
               OR raised_ts <  timestamp'2025-01-01'
               OR raised_ts >= timestamp'2025-06-01' THEN 'raised_ts_out_of_range'
             WHEN closed_ts < raised_ts THEN 'closed_before_raised'
        END""")))

t.filter("reject_reason IS NOT NULL").write.mode("overwrite").saveAsTable(f"{S}.silver_rejects")
t.groupBy("reject_reason").count().show()               # 500 / 700 / 900 (NULL = the good rows)

# Decision 4: define resolution and breach once
WIN_END = "timestamp'2025-06-29 23:59:59'"
silver = (t.filter("reject_reason IS NULL")
  .drop("reject_reason", "closed_raw")
  .withColumn("resolution_hours", F.expr(
      "CASE WHEN closure_known THEN (unix_timestamp(closed_ts) - unix_timestamp(raised_ts))/3600 END"))
  .withColumn("sla_breach", F.expr(f"""
      CASE WHEN closure_known THEN resolution_hours > 168
           WHEN status = 'OPEN' THEN raised_ts <= {WIN_END} - INTERVAL 168 HOURS
      END""")))
silver.write.mode("overwrite").saveAsTable(f"{S}.silver_complaint")

after dedupe: 60500
closed_ts lost to NULL: 1200
+--------------------+-----+
|       reject_reason|count|
+--------------------+-----+
|                NULL|58400|
|closed_before_raised|  900|
|raised_ts_out_of_...|  700|
|        unknown_ward|  500|
+--------------------+-----+



In [0]:
s = spark.table(f"{S}.silver_complaint")
print("Silver rows:", s.count())                                            # 58,400
print("CLOSED + closure_known=false:",
      s.filter("status='CLOSED' AND NOT closure_known").count())            # 1,200
print("in both that set and the open set:",
      s.filter("status='CLOSED' AND NOT closure_known AND status='OPEN'").count())  # 0
print("negative durations:", s.filter("resolution_hours < 0").count())      # 0
print("closure_known = true:", s.filter("closure_known").count())           # note this number for Gold

s.filter("closure_known").groupBy("category") \
 .agg(F.expr("percentile(resolution_hours, 0.5)").alias("median_h")) \
 .orderBy("median_h").show()

Silver rows: 58400
CLOSED + closure_known=false: 1200
in both that set and the open set: 0
negative durations: 0
closure_known = true: 48325
+-------------+------------------+
|     category|          median_h|
+-------------+------------------+
|Stray Animals|12.108333333333334|
|  Streetlight|            17.975|
|      Garbage|30.283333333333335|
| Water Supply|48.608333333333334|
| Property Tax| 72.90833333333333|
|     Drainage| 95.71666666666667|
|  Road Repair|           240.025|
| Encroachment| 357.0833333333333|
+-------------+------------------+

